In [1]:
# Install necessary libraries
# !pip install tensorflow pandas nltk scikit-learn

# Import libraries
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
import nltk
from nltk import pos_tag, word_tokenize
import re
import numpy as np
import pandas as pd

# Download NLTK datasets
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

# Step 1: Define synthetic dataset
def load_data():
    data = {
        'text': [
            "The quick brown fox jumps over the lazy dog.",
            "This is a sample text for the first user.",
            "Here is another sentence by the first user.",
            "The weather today is quite sunny and warm.",
            "User two prefers a different writing style.",
            "This sentence has no relation to the others.",
            "Another sentence for user two's writing.",
            "Unauthorized user trying to access the system.",
            "This is a sample text from an unknown user.",
            "Unauthorized user input goes in this field.",
            "Another unauthorized user writing here.",
            "Another authorized user writes here.",
            "User three has a distinct writing style.",
            "User three writes something else here.",
            "Authorized user three typing another sentence.",
            "Random text that unauthorized users might enter.",
            "Different writing pattern seen in this text.",
            "User four adding his own sample text here.",
            "Another text sample from user four.",
            "Unauthorized user attempting access.",
        ],
        'user_id': [
            "user1", "user1", "user1", "user1",
            "user2", "user2", "user2", 
            "user3", "user3", "user3", 
            "user4", "user4", "user4",
            "user5", "user5", "user5",
            "user6", "user6", "user6", 
            "user7"
        ],
        'label': [
            1, 1, 1, 1,  # User 1 (authorized)
            1, 1, 1,     # User 2 (authorized)
            0, 0, 0,     # Unauthorized users
            1, 1, 1,     # User 3 (authorized)
            0, 0, 0,     # Unauthorized
            1, 1, 1,     # User 4 (authorized)
            0            # Unauthorized
        ]
    }
    df = pd.DataFrame(data)
    return df['text'], df['label']  # Return text and label for binary classification

# Step 2: Preprocessing function for stylometric features
def preprocess_text(text):
    # Tokenize the text
    tokens = word_tokenize(text)
    
    # Get POS tags
    pos_tags = [tag for word, tag in pos_tag(tokens)]
    
    # Calculate punctuation frequency
    punct_freq = len(re.findall(r'[^\w\s]', text)) / len(text)
    
    # Calculate average word length
    avg_word_len = np.mean([len(word) for word in tokens])
    
    return ' '.join(tokens), punct_freq, avg_word_len, pos_tags

# Step 3: Tokenizing and padding sequences
def prepare_data(texts, max_len=100):
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(texts)
    sequences = tokenizer.texts_to_sequences(texts)
    padded = pad_sequences(sequences, maxlen=max_len, padding='post')
    
    return padded, tokenizer

# Step 4: Define the RNN model
def build_rnn(vocab_size, max_len, num_classes=1):
    model = Sequential()
    model.add(Embedding(input_dim=vocab_size, output_dim=128, input_length=max_len))
    model.add(LSTM(128, return_sequences=True))
    model.add(Dropout(0.5))
    model.add(LSTM(128))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(num_classes, activation='sigmoid'))
    
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# Step 5: Function to classify user input
def classify_input(user_input, model, tokenizer, max_len):
    # Preprocess the input text
    processed_input, punct_freq, avg_word_len, pos_tagging = preprocess_text(user_input)
    
    # Tokenize and pad the input sequence
    input_sequence = tokenizer.texts_to_sequences([processed_input])
    padded_input = pad_sequences(input_sequence, maxlen=max_len, padding='post')
    
    # Predict the label (authorized or unauthorized)
    prediction = model.predict(padded_input)[0][0]
    
    # Return the result based on threshold (0.5 for binary classification)
    if prediction >= 0.7:
        return "Access Granted"
    else:
        return "Access Denied"

# Main script
if __name__ == "__main__":
    # Load the synthetic dataset
    texts, labels = load_data()
    
    # Step 6: Preprocess text for stylometric features
    processed_texts, punct_freqs, avg_word_lens, pos_tags = [], [], [], []
    for text in texts:
        processed_text, punct_freq, avg_word_len, pos_tagging = preprocess_text(text)
        processed_texts.append(processed_text)
        punct_freqs.append(punct_freq)
        avg_word_lens.append(avg_word_len)
        pos_tags.append(pos_tagging)
    
    # Prepare sequences for the RNN
    max_len = 100
    X, tokenizer = prepare_data(processed_texts, max_len=max_len)
    
    # Add stylometry features such as punctuation frequency and average word length
    stylometry_features = np.column_stack((punct_freqs, avg_word_lens))
    
    # Step 7: Split the dataset into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.2, random_state=42)
    
    # Step 8: Build and train the RNN model
    vocab_size = len(tokenizer.word_index) + 1
    model = build_rnn(vocab_size, max_len)
    
    # Train the model
    model.fit(X_train, y_train, epochs=5, batch_size=8, validation_split=0.2)
    
    # Evaluate the model
    loss, accuracy = model.evaluate(X_test, y_test)
    print(f'Test Accuracy: {accuracy:.2f}')
    
    # Step 9: Prompt user for input and classify
    user_input = input("Enter text for classification: ")
    result = classify_input(user_input, model, tokenizer, max_len)
    print(result)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\conmy\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\conmy\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
c:\Users\conmy\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 10s 625ms/step - accuracy: 0.5000 - loss: 0.6927 - val_accuracy: 0.5000 - val_loss: 0.6944
Epoch 2/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - accuracy: 0.6944 - loss: 0.6755 - val_accuracy: 0.5000 - val_loss: 0.7036
Epoch 3/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 248ms/step - accuracy: 0.6111 - loss: 0.6706 - val_accuracy: 0.5000 - val_loss: 0.7175
Epoch 4/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step - accuracy: 0.6944 - loss: 0.6345 - val_accuracy: 0.5000 - val_loss: 0.7674
Epoch 5/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step - accuracy: 0.6944 - loss: 0.6210 - val_accuracy: 0.5000 - val_loss: 0.7795
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.7500 - loss: 0.5687
Test Accuracy: 0.75
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 334ms/step
Access Denied
